# 02 · Document Grounding

**목표**: 공개 help.sap.com을 읽기 전용으로 질의하는 관리형 RAG와, Custom vector repository를 사용하는 문서 Grounding의 차이를 관찰합니다.

Grounding은 관련 문서 청크를 먼저 찾고 결과를 prompt에 넣어 근거 없는 답변을 줄입니다. 답변뿐 아니라 서비스가 반환한 grounding 메타데이터를 확인해 근거를 점검합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">Track B: Custom Repository Grounding (상태 변경)</h2>
<p style="margin: 7px 0;">Repository ID를 입력하면 해당 Custom vector 저장소의 문서를 목록으로 보고 .txt/.md를 업로드·삭제한 뒤 Grounding 질의를 실행할 수 있습니다. 이 UI의 업로드·삭제와 Repository ID 저장은 실제 상태를 변경하므로 개인 또는 교육용 repository만 사용하세요.</p>
<p style="margin: 7px 0;"><strong>순서:</strong> 업로드 → 인덱싱 대기 → 새로고침으로 문서 확인 → 질의 → 필요 시 Grounding 인덱스 문서 삭제입니다. 업로드 직후에는 아직 검색되지 않을 수 있으며, 삭제해도 Object Storage 원본은 남을 수 있습니다.</p>
  </div>
</div>


In [2]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]
from src.grounding.ui import build_grounding_ui

grounding_ui = build_grounding_ui()
grounding_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', height=900)


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>교육용 repository에서만 업로드·삭제를 수행하고, 인덱싱 완료 뒤 문서 목록을 새로고침한다.</li>
<li>응답의 <code>grounding.data</code>에 서비스가 반환한 문서·chunk·citation 메타데이터가 있는지 확인한다. 없으면 citation을 추정하지 않는다.</li>
<li>아래 Track A 코드는 공개 help.sap.com을 읽기 전용으로 질의하는 선택 심화 자료다.</li>
</ul>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">Track A: Public Help Grounding (읽기 전용)</h2>
<p style="margin: 7px 0;">아래 셀은 공개 help.sap.com만 대상으로 검색·context 주입·답변 생성을 확인합니다. Custom repository의 문서 업로드·삭제나 .env 변경은 수행하지 않습니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Grounding pipeline 구성</h2>
<p style="margin: 7px 0;">DocumentGroundingFilter는 검색 위치를 정합니다. DocumentGroundingPlaceholders는 질문 input과 검색 결과 output을 template에 연결합니다. grounding_response는 Grounding 모듈이 자동으로 채웁니다.</p>
  </div>
</div>


In [6]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]
from lab_common.config import load_settings
from src.grounding.lab_core import run_help_grounded_query, stream_help_grounded_query

settings = load_settings()
print("=" * 56)
print("Step 4 · 공개 그라운딩 + 스트리밍")
print("=" * 56)

Step 4 · 공개 그라운딩 + 스트리밍


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 근거 기반 질의</h2>
<p style="margin: 7px 0;">service.run에는 질문인 user_query만 전달합니다. 내부적으로 검색, context 주입, LLM 답변이 한 번에 수행됩니다.</p>
  </div>
</div>


In [7]:
# 1) 그라운딩 질의
print("\n[1] 그라운딩 질의: 'What is Joule?'")
answer = run_help_grounded_query("What is Joule?", settings)
print("  응답:", answer[:300])


[1] 그라운딩 질의: 'What is Joule?'
  응답: Joule is SAP’s generative AI copilot integrated across SAP’s cloud enterprise portfolio. It is designed to understand your business and revolutionize interactions with SAP business systems, simplifying tasks and optimizing each user touch-point.

Joule provides informational, navigational, and trans


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. Streaming</h2>
<p style="margin: 7px 0;">run은 완성된 답을 한 번에 받지만 stream은 생성 조각을 순서대로 받습니다. delta.content는 이번 chunk에서 새로 추가된 텍스트입니다.</p>
  </div>
</div>


In [8]:
# 2) 스트리밍
print("\n[2] 스트리밍: 'What is SAP AI Core?'")
chunks = 0
print("  ", end="")
for delta in stream_help_grounded_query("What is SAP AI Core?", settings):
    chunks += 1
    print(delta, end="", flush=True)
print(f"\n  (총 {chunks}개 청크 수신)")


[2] 스트리밍: 'What is SAP AI Core?'
  SAP AI Core is a service within SAP Business Technology Platform designed to manage the execution and operations of AI assets in a standardized, scalable, and hyperscaler-agnostic way. It supports full lifecycle management of AI scenarios, including the creation, training, deployment, and monitoring of machine learning models. SAP AI Core provides seamless integration with SAP solutions and enables the use of open-source frameworks for AI functions.

Key features of SAP AI Core include:
- Execution of pipelines as batch jobs for tasks like data preprocessing, model training, or batch inference.
- Deployment of trained machine learning models as web services to serve inference requests.
- Management of ML artifacts and workflows (model training, metrics tracking, data, models, and deployments) through a uniform API lifecycle.
- Multitenancy support for tenant-aware applications and asset segregation.
- Integration with cloud infrastructure (Docker reg

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. Grounding 전후 비교</h2>
<p style="margin: 7px 0;">LLM만 사용한 답과 같은 질문을 비교해 보세요. template에 Context에 없으면 문서에 없음이라고 답하라는 규칙을 추가하면 근거 범위를 더 명확히 제한할 수 있습니다.</p>
  </div>
</div>


In [9]:
# 3) Step 1 환각 질문 재질의 → 비교
print("\n[3] 환각 비교: 'How does the data masking of the orchestration service work?'")
answer = run_help_grounded_query("How does the data masking of the orchestration service work?", settings)
print("  응답(그라운딩 있음):", answer[:400])

print("\n✅ Step 4 완료 — 그라운딩/스트리밍 동작. [3]을 Step 1 [3]과 비교하세요.")


[3] 환각 비교: 'How does the data masking of the orchestration service work?'
  응답(그라운딩 있음): Based on the provided context, the orchestration service supports a Data Masking module, which works as follows:

- **Function:** The Data Masking module hides or masks sensitive information during the orchestration process.
- **Placement:** It is one of the modular pre-processing or post-processing steps in the orchestration pipeline. You can configure this module along with others (like Input Fi

✅ Step 4 완료 — 그라운딩/스트리밍 동작. [3]을 Step 1 [3]과 비교하세요.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 5</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">답변과 Grounding 근거 메타데이터 함께 읽기</h2>
<p style="margin: 7px 0;">관리형 Grounding은 서비스 버전과 repository 유형에 따라 문서 ID, chunk, URL, citation 등을 metadata로 반환할 수 있습니다. 아래 결과의 <code>grounding.data</code>를 확인하세요. metadata가 비어 있다면 출처를 추정하거나 만들어서는 안 됩니다.</p>
  </div>
</div>


In [ ]:
import json
from src.grounding.lab_core import run_help_grounded_query_with_trace

traced_result = run_help_grounded_query_with_trace('What is Joule?', settings)
print('답변:')
print(traced_result['answer'])
print()
print('Grounding metadata:')
print(json.dumps(traced_result['grounding'], ensure_ascii=False, indent=2))

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 6</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">답변 가능 범위를 점검하기</h2>
<p style="margin: 7px 0;">문서에 있는 질문, 문서와 무관한 질문, 일부만 근거가 있을 질문을 비교합니다. 각 응답에서 답변이 문맥 부족을 명시하는지와 metadata가 반환됐는지를 직접 판정하세요. 문자열에 특정 거절 문구가 있는지만으로 품질을 판단하지는 않습니다.</p>
  </div>
</div>


In [ ]:
grounding_checks = [
    ('근거 있음', 'What is SAP AI Core?', '문서 근거로 답하는지 확인'),
    ('근거 부족', 'What is the capital of Korea?', '문맥 부족을 말하는지 확인'),
    ('부분 근거', 'Does SAP AI Core support model training and what was its acquisition price?', '가격 정보의 근거 부족을 분리하는지 확인'),
]
for category, question, expected in grounding_checks:
    outcome = run_help_grounded_query_with_trace(question, settings)
    metadata = outcome['grounding']
    print(f'[{category}] {question}')
    print(f'판정 기준: {expected}')
    print(f'metadata 반환: {metadata["metadata_available"]}')
    print(f'답변: {outcome["answer"][:500]}')
    print()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 7</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">Streaming은 첫 응답 체감을 개선하는가</h2>
<p style="margin: 7px 0;">Streaming은 답변 품질을 높이는 기능이 아니라, 첫 텍스트가 보일 때까지의 체감 대기 시간을 줄이는 전송 방식입니다. 첫 chunk와 전체 완료까지의 시간을 관찰하세요.</p>
  </div>
</div>


In [ ]:
from time import perf_counter

started_at = perf_counter()
first_chunk_at = None
chunks = []
for delta in stream_help_grounded_query('What is SAP AI Core?', settings):
    if first_chunk_at is None:
        first_chunk_at = perf_counter()
    chunks.append(delta)
finished_at = perf_counter()
print(f'첫 chunk: {(first_chunk_at or finished_at) - started_at:.2f}초')
print(f'전체 완료: {finished_at - started_at:.2f}초')
print(f'수신 chunk: {len(chunks)}개')
print('응답 앞부분:', ''.join(chunks)[:300])

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ Track B 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">Custom repository 수명주기 검증</h2>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>교육용 repository에 출처가 명확한 짧은 문서를 하나 업로드합니다.</li><li>인덱싱 완료 후 새로고침하여 document ID를 확인하고, 문서에만 있는 질문을 합니다.</li><li>응답의 grounding metadata에 반환된 source/chunk/citation을 확인합니다. metadata가 없으면 답변만으로 출처를 단정하지 않습니다.</li><li>문서를 삭제하고 동일 질문을 재실행하여 검색 결과가 사라지거나 근거 부족으로 전환되는지 확인합니다.</li><li>업로드한 문서 안의 지시문은 신뢰할 수 없는 데이터입니다. 시스템 규칙을 바꾸거나 문서 범위 밖 행동을 유도하는 내용은 답변 지시로 따르지 않아야 합니다.</li></ol>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 미션</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">Grounding 평가셋 만들기</h2>
<p style="margin: 7px 0;">근거 있는 질문, 근거 부족 질문, 부분 근거 질문을 각각 포함해 6개 이상의 평가셋을 만드세요. 답변 적합성, 적절한 근거 부족 응답, metadata 내 기대 출처 확인 여부를 기록합니다. 이 노트북은 답변 품질을 자동으로 확정하지 않으며, 사람이 결과와 근거를 검토합니다.</p>
  </div>
</div>
